# CAS Exam 5: Testing the Reasonableness of Reserve Estimates

**Source:** Friedland, J. *Estimating Unpaid Claims Using Basic Techniques*, CAS, 2010 — Chapter 21

**Exam task:** B — Evaluate the reasonableness of reserve estimates; test assumptions of development methods

**Learning goals:**
1. Explain why testing reserve estimates is a required step in the reserving process
2. Apply the **hindsight development test** to evaluate prior reserve adequacy
3. Compare reserve estimates across multiple methods to identify unexplained divergence
4. Apply a **calendar year emergence test** (one-period rollforward) to test development assumptions
5. Use industry benchmarks and diagnostic ratios to sanity-check reserve levels
6. Articulate the diagnostic questions Friedland poses when evaluating reserve estimates

> **See also:** `exam5_full_reserving_analysis.ipynb` for the full reserve selection workflow; `exam5_fullworkflow_methodcomparison.ipynb` for Actual vs Expected one-period tests.

## Formula Sheet Quick Reference

| Test | Formula / Approach |
|---|---|
| **Hindsight development** | Apply current LDFs to prior-year reserves; compare projected ultimates to current ultimates |
| **Hindsight reserve ratio** | Projected ultimate (from prior reserve) / Current ultimate estimate |
| **One-period emergence test** | Expected emergence = Prior ultimate estimate × (1 − 1/LDF) × (1/prior CDF); compare to actual |
| **Reserve to premium ratio** | IBNR / Earned premium — compare to industry for the line |
| **Reserve to reported loss ratio** | IBNR / Reported losses — compare to prior years and industry |
| **% Reported** | 1 / CDF(current age → ultimate) — is the maturity consistent with expected? |

## Section 1: Why Testing Is Required

A reserving actuary does not simply apply a method and report the result. Friedland emphasizes that **evaluating the reasonableness** of the selected reserve is as important as the projection itself.

**Reasons for testing:**
1. **Model error** — development methods assume past patterns will continue. Testing reveals whether this assumption has held in recent periods.
2. **Data quality** — errors, restatements, or categorization changes in the underlying triangle may distort projections. Tests can surface these.
3. **Environmental changes** — claims handling, legal, or underwriting changes may invalidate historical LDFs. Testing identifies divergence.
4. **Audit and communication** — management, auditors, and regulators require the actuary to demonstrate that the selected reserve is reasonable, not just the output of a formula.
5. **Professional standards** — ASOP 43 requires the actuary to consider the reasonableness of the unpaid claim estimate relative to alternative estimates and available information.

**Key principle from Friedland:**
> *The actuary should be able to explain what is driving any significant differences between methods and between the current estimate and prior estimates. Unexplained divergence is a red flag.*

### The Diagnostic Framework

Testing reserve estimates involves three categories:

| Category | Key Question | Test |
|---|---|---|
| **Consistency over time** | Are current ultimates consistent with prior estimates? | Hindsight development test |
| **Consistency across methods** | Do different methods agree? Can divergence be explained? | Method comparison |
| **Reasonableness vs. external benchmarks** | Are reserves reasonable relative to the book size and line? | Reserve/premium, reserve/reported ratios |

## Section 2: The Hindsight Development Test

The **hindsight development test** (also called the "development of IBNR" test) evaluates whether prior reserve estimates were adequate by applying development factors to those prior reserves and comparing to the current estimate.

### Procedure

1. Take the **prior-year reserve** (IBNR) for each accident year at the prior valuation date
2. Apply **one period's worth of development** to the prior reserve — this is what we would have projected would happen if the prior reserve was correct
3. Compare the projected result to the **actual emergence** in the period
4. The ratio of actual to expected emergence (or actual to expected reserve run-off) tells you whether the prior reserve was over or under

**Alternative formulation:** Apply current CDFs to prior-year case-incurred reserves at each accident year's age.

### Worked Example

At year-end 2023, the actuary estimated the following ultimates (prior estimates):

| AY | Reported at 12/31/23 | Prior Ultimate Estimate |
|---|---|---|
| 2021 | \$5,400 | \$6,200 |
| 2022 | \$4,800 | \$6,100 |
| 2023 | \$3,600 | \$6,000 |

At year-end 2024, the reported losses and current ultimate estimates are:

| AY | Reported at 12/31/24 | Current Ultimate Estimate |
|---|---|---|
| 2021 | \$5,900 | \$6,300 |
| 2022 | \$5,500 | \$6,500 |
| 2023 | \$4,700 | \$6,400 |

**Test:** Prior IBNR vs. Actual Emergence

| AY | Prior IBNR | Actual Emergence | Remaining IBNR | Prior IBNR (Expected Total) | Adequacy |
|---|---|---|---|---|---|
| 2021 | \$800 | \$500 | \$400 | \$800 (prior) vs. \$900 (actual needed) | Under by \$100 |
| 2022 | \$1,300 | \$700 | \$1,000 | \$1,300 vs. \$1,700 needed | Under by \$400 |
| 2023 | \$2,400 | \$1,100 | \$1,700 | \$2,400 vs. \$2,800 needed | Under by \$400 |

**Interpretation:** Prior-year reserves were systematically inadequate. All accident years are showing negative redundancy — the prior estimates are proving to be lower than current estimates. This pattern suggests the actuary should:
1. Investigate whether LDFs were too low in prior year
2. Check for a systematic environmental change (worsening claims experience)
3. Consider whether the selected method is appropriate

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Prior year-end data (12/31/2023)
prior = pd.DataFrame({
    'ay': [2021, 2022, 2023],
    'reported_prior': [5_400, 4_800, 3_600],
    'ultimate_prior': [6_200, 6_100, 6_000],
})
prior['ibnr_prior'] = prior['ultimate_prior'] - prior['reported_prior']

# Current year-end data (12/31/2024)
current = pd.DataFrame({
    'ay': [2021, 2022, 2023],
    'reported_current': [5_900, 5_500, 4_700],
    'ultimate_current': [6_300, 6_500, 6_400],
})
current['ibnr_current'] = current['ultimate_current'] - current['reported_current']

# Merge and compute hindsight metrics
df = prior.merge(current, on='ay')
df['emergence'] = df['reported_current'] - df['reported_prior']
df['ultimate_change'] = df['ultimate_current'] - df['ultimate_prior']
df['adequacy'] = -df['ultimate_change']  # positive = redundant (prior was too high)

print("Hindsight Development Test:")
display(df[['ay','ibnr_prior','emergence','ibnr_current','ultimate_change','adequacy']].rename(columns={
    'ay': 'AY', 'ibnr_prior': 'Prior IBNR',
    'emergence': 'Actual Emergence',
    'ibnr_current': 'Current IBNR',
    'ultimate_change': 'Ult Change (Deficiency)',
    'adequacy': 'Redundancy (Deficiency if −)'
}).set_index('AY'))

print(f"\nTotal Prior IBNR: ${df['ibnr_prior'].sum():,.0f}")
print(f"Total Ultimate Change (Deficiency): ${df['ultimate_change'].sum():,.0f}")
print(f"Direction: {'Consistent development of deficiency → reserves were inadequate' if df['ultimate_change'].sum() > 0 else 'Consistent redundancy'}")

# Waterfall chart: prior IBNR → emergence → remaining
fig, ax = plt.subplots(figsize=(9, 4))
x = ['AY 2021', 'AY 2022', 'AY 2023']
width = 0.3
x_pos = np.arange(len(x))
ax.bar(x_pos - width/2, df['ibnr_prior'], width, label='Prior IBNR', color='steelblue', alpha=0.8)
ax.bar(x_pos + width/2, df['ibnr_current'] + df['emergence'], width,
       label='Needed (Current IBNR + Actual Emergence)', color='tomato', alpha=0.8)
ax.set_xticks(x_pos); ax.set_xticklabels(x)
ax.set_ylabel('Losses ($000s)')
ax.set_title('Hindsight Test: Prior IBNR vs. Total Needed')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 3: Comparing Reserve Estimates Across Methods

The actuary should **never rely on a single method** for reserve estimation. Comparing methods is both a best practice and required by professional standards (ASOP 43). Key principles:

### When Methods Should Agree

| Scenario | Expected Result |
|---|---|
| Stable, mature accident years with predictable development | All methods should produce similar estimates |
| BF and chain ladder when a priori is well-calibrated | BF ≈ chain ladder for mature years; BF < chain ladder for immature years with adverse development |
| Frequency-severity and chain ladder on a stable book | Should converge for mature years |

### When Methods Diverge — Red Flags

| Divergence Pattern | Possible Cause |
|---|---|
| **CL >> BF** for recent accident years | CL is picking up adverse development in immature years; BF's a priori may be more appropriate |
| **CL << BF** for mature accident years | Chain ladder says IBNR is low; BF's a priori suggests more is needed → possible favorable development distorting the triangle |
| **Paid CL >> Incurred CL** | Case reserves are redundant; or paid development is understated |
| **Paid CL << Incurred CL** | Case reserves are deficient; expected to develop upward |
| **Expected claims method >> development method** | Historical LDFs may be too low (favorable development bias) |
| **Methods wide range** for recent years | Normal for immature accident years; problematic if same for mature years |

### The Method Comparison Diagnostic

For each accident year, compute the **coefficient of variation** across methods:

$$\text{CV}_{AY} = \frac{\text{Std Dev of ultimate estimates}}{\text{Mean of ultimate estimates}}$$

- CV < 5% → methods are in close agreement → high confidence in selected ultimate
- CV 5–15% → moderate divergence → requires judgment and explanation
- CV > 15% → high divergence → investigate before selecting; may indicate structural data problems

In [ ]:
import pandas as pd
import numpy as np

# Method comparison table: multiple estimates for each AY
methods = ['Chain Ladder (Paid)', 'Chain Ladder (Incurred)', 'BF (Paid)', 'Expected Claims', 'Selected']

estimates = pd.DataFrame({
    'Chain Ladder (Paid)':    [6_280, 6_420, 6_350, 6_510],
    'Chain Ladder (Incurred)':[6_300, 6_500, 6_450, 6_600],
    'BF (Paid)':              [6_250, 6_380, 6_300, 6_450],
    'Expected Claims':        [6_350, 6_350, 6_350, 6_350],
}, index=[2021, 2022, 2023, 2024])

estimates.index.name = 'AY'

# Compute dispersion metrics
estimates['Mean'] = estimates.mean(axis=1)
estimates['Std Dev'] = estimates.std(axis=1)
estimates['CV'] = estimates['Std Dev'] / estimates['Mean']
estimates['Selected'] = [6_300, 6_450, 6_400, 6_530]  # actuary's selected
estimates['Selected vs Mean'] = estimates['Selected'] / estimates['Mean'] - 1

print("Method Comparison — Ultimate Estimates:")
display(estimates[['Chain Ladder (Paid)', 'Chain Ladder (Incurred)', 'BF (Paid)',
                    'Expected Claims', 'Mean', 'CV', 'Selected', 'Selected vs Mean']].style.format({
    'Mean': '${:,.0f}', 'Std Dev': '${:,.0f}', 'CV': '{:.1%}',
    'Selected vs Mean': '{:+.1%}',
    'Chain Ladder (Paid)': '${:,.0f}', 'Chain Ladder (Incurred)': '${:,.0f}',
    'BF (Paid)': '${:,.0f}', 'Expected Claims': '${:,.0f}', 'Selected': '${:,.0f}',
}))

print("\nInterpretation:")
for ay in estimates.index:
    cv = estimates.loc[ay, 'CV']
    sel_vs_mean = estimates.loc[ay, 'Selected vs Mean']
    flag = '✅' if cv < 0.05 else ('⚠️' if cv < 0.15 else '🔴')
    print(f"  AY {ay}: CV={cv:.1%} {flag}; Selected is {sel_vs_mean:+.1%} vs mean")

## Section 4: Calendar Year Emergence Test (One-Period Rollforward)

The **calendar year emergence test** (or one-period rollforward) compares:
- **Expected emergence**: what the prior ultimate estimates predicted would emerge during the year
- **Actual emergence**: what actually emerged in the triangle during the year

A systematic ratio of actual to expected > 1.0 across accident years indicates reserves are **developing adversely** (inadequate). Ratio < 1.0 indicates favorable development.

### Formula

For accident year $w$ moving from age $d$ to $d+12$:

$$\text{Expected Emergence} = \text{Prior Ultimate} \times \left(\frac{1}{\text{CDF}(d)} - \frac{1}{\text{CDF}(d+12)}\right)$$

Where:
- $1/\text{CDF}(d)$ = % paid (or reported) at prior evaluation age
- $1/\text{CDF}(d+12)$ = % paid (or reported) at current evaluation age
- Difference = expected % of ultimate to emerge in the period
- Multiplied by prior ultimate = expected dollar emergence

**Actual vs. Expected Ratio (A/E):**

$$A/E = \frac{\text{Actual Emergence}}{\text{Expected Emergence}}$$

- A/E consistently > 1.0 → adverse development; LDFs may be understated
- A/E consistently < 1.0 → favorable development; LDFs may be overstated  
- A/E ≈ 1.0 with some variability → development is tracking as expected

### Key Diagnostic Questions (Friedland Ch 21)

Friedland suggests the actuary ask these questions when evaluating reserve adequacy:

1. Is the selected development pattern consistent with the most recent data?
2. Are the latest development factors higher or lower than the selected factors — and why?
3. How do the selected ultimates compare to the prior year estimates after one year of development?
4. Does the calendar year diagnostic show a systematic trend (diagonal effects)?
5. Are the methods in reasonable agreement? Can divergence be explained?
6. Does the reserve-to-premium ratio compare favorably to industry data?
7. Are there any large individual claims that are driving development?
8. Have there been any changes in claims handling, settlement patterns, or reporting?
9. Are the results sensitive to the tail factor selection?
10. Do the expected claims (BF a priori) make sense given current pricing?

In [ ]:
import pandas as pd
import numpy as np

# One-period rollforward (Actual vs. Expected emergence)
# Prior valuation: 12/31/2023; Current: 12/31/2024

# CDFs to ultimate (from the selected development pattern)
cdfs = {12: 3.200, 24: 1.950, 36: 1.380, 48: 1.120, 60: 1.040, 72: 1.010, 84: 1.000}

rollforward = pd.DataFrame({
    'ay': [2021, 2022, 2023, 2024],
    'prior_age': [36, 24, 12, 0],   # months at 12/31/2023
    'current_age': [48, 36, 24, 12], # months at 12/31/2024
    'prior_ultimate': [6_200, 6_100, 6_000, None],
    'actual_emergence': [500, 700, 1_100, 1_620],  # actual change in paid losses
})

# For AY 2024 (no prior): skip (not enough history)
# For others: compute expected emergence
def expected_emerg(row):
    if pd.isna(row['prior_ultimate']) or row['prior_age'] == 0:
        return np.nan
    pct_prior = 1 / cdfs.get(row['prior_age'], 1.0)
    pct_current = 1 / cdfs.get(row['current_age'], 1.0)
    return row['prior_ultimate'] * (pct_current - pct_prior)

rollforward['expected_emergence'] = rollforward.apply(expected_emerg, axis=1)
rollforward['ae_ratio'] = rollforward['actual_emergence'] / rollforward['expected_emergence']

print("One-Period Rollforward — Actual vs. Expected Emergence:")
display(rollforward[['ay', 'prior_age', 'current_age', 'prior_ultimate',
                       'expected_emergence', 'actual_emergence', 'ae_ratio']].rename(columns={
    'ay': 'AY', 'prior_age': 'Prior Age', 'current_age': 'Current Age',
    'prior_ultimate': 'Prior Ultimate', 'expected_emergence': 'Expected Emergence',
    'actual_emergence': 'Actual Emergence', 'ae_ratio': 'A/E Ratio'
}).set_index('AY').dropna())

ae_ratios = rollforward['ae_ratio'].dropna()
print(f"\nAverage A/E Ratio: {ae_ratios.mean():.3f}")
if ae_ratios.mean() > 1.05:
    print("Interpretation: Adverse development — actual emergence is consistently higher than expected.")
    print("Implication: Selected development factors may be understated; reserves may be inadequate.")
elif ae_ratios.mean() < 0.95:
    print("Interpretation: Favorable development — actual emergence is below expected.")
    print("Implication: Selected development factors may be overstated; reserves may be redundant.")
else:
    print("Interpretation: Development tracking within expected range.")

## Section 5: Benchmark Ratios and External Comparisons

### Reserve-to-Premium and Reserve-to-Loss Ratios

Two simple but powerful sanity checks:

**Reserve-to-Premium Ratio:**

$$\text{R/P Ratio} = \frac{\text{Total Loss Reserve}}{\text{Earned Premium}}$$

Compare to:
- The insurer's own historical ratios (are reserves growing relative to premium?)
- Industry data for the same line (ISO, NCCI, AM Best aggregates)

A ratio much higher than industry may signal reserve deficiency; much lower may signal overstatement or a very short-tailed book.

**Reported-to-Incurred Ratio (% Developed):**

$$\% \text{Developed} = \frac{\text{Cumulative Reported}}{\text{Selected Ultimate}} = \frac{1}{\text{CDF to ultimate at current age}}$$

Compare across accident years: is the most recent accident year at a development % consistent with prior years at the same age? Unusually low % may indicate the triangle is thinning due to claim acceleration or mix changes.

### Using Schedule P for Benchmarking

The **NAIC Schedule P** provides 10 years of development data by line of business for every US insurer. The actuary can:
1. Read off the 10-year development pattern and compare to selected LDFs
2. Compute reserve-to-premium ratios from Schedule P data and compare to industry
3. Identify calendar year diagonal effects in Schedule P (reserve strengthening waves)
4. Compare the insurer's reserve development to industry average development

### Summary: A Reserve Review Checklist

Before finalizing a reserve opinion, the actuary should be able to answer **yes** to each of these:

| Check | Question |
|---|---|
| ✅ Hindsight | Are current ultimates broadly consistent with prior estimates? Is any deficiency explained? |
| ✅ Method agreement | Do paid and incurred development methods agree? Is BF close to chain ladder for mature years? |
| ✅ A/E test | Is the A/E ratio on the one-period rollforward near 1.0? |
| ✅ Benchmarks | Is the reserve/premium ratio consistent with industry and with prior years? |
| ✅ Tail | Is the tail factor selection reasonable given external benchmarks? |
| ✅ Diagnostics | Have calendar year effects, large losses, and environmental changes been addressed? |
| ✅ ASOP 43 | Has the range of estimates and key uncertainties been communicated? |

## Section 6: Practice Problems

### Problem 1 — Hindsight Development Test

At year-end 2023, an actuary selected the following ultimates:

| AY | Reported at 12/31/23 | Selected Ultimate |
|---|---|---|
| 2021 | \$8,200 | \$9,100 |
| 2022 | \$7,500 | \$9,500 |

At year-end 2024, the reported losses are:

| AY | Reported at 12/31/24 | New Selected Ultimate |
|---|---|---|
| 2021 | \$8,900 | \$9,400 |
| 2022 | \$8,400 | \$10,200 |

**(a)** Calculate the prior IBNR for each accident year.

**(b)** Calculate the actual emergence (change in reported) for each AY.

**(c)** Calculate the remaining IBNR at year-end 2024.

**(d)** Did the prior reserves prove adequate? Compute the deficiency or redundancy for each AY.

**(e)** The pattern is consistent across both AYs. What does this suggest about the prior reserve selection?

---

### Problem 2 — A/E Rollforward

An actuary uses the following CDFs in their prior selection:

| Age | CDF to Ultimate |
|---|---|
| 12 | 4.000 |
| 24 | 2.400 |
| 36 | 1.600 |

At 12/31/2023, AY 2022 had reported losses of \$3,000 at age 24, with selected ultimate = \$7,200.

During 2024, AY 2022 (now at age 36) had actual emergence of \$1,950.

**(a)** Calculate the expected emergence from age 24 to age 36.

**(b)** Calculate the A/E ratio.

**(c)** Is development favorable or adverse? What should the actuary investigate?

---

### Problem 3 — Benchmark and Diagnostic

**(a)** An insurer's general liability reserve-to-earned-premium ratio has increased from 0.85 to 1.15 over three years. The industry average is 0.90. List three explanations for this increase and describe how you would investigate each.

**(b)** A chain ladder on incurred losses produces an ultimate of \$12.5M for AY 2023. The BF method produces \$10.8M. The a priori for BF was set at the current rate level. AY 2023 is at 24-month development (CDF to ultimate = 2.5). Which estimate do you give more weight to, and why?

---

### Solutions

<details>
<summary>Click to reveal solutions</summary>

**Problem 1:**  
(a) Prior IBNR: AY 2021 = $9,100 − $8,200 = **$900**; AY 2022 = $9,500 − $7,500 = **$2,000**  
(b) Actual emergence: AY 2021 = $8,900 − $8,200 = **$700**; AY 2022 = $8,400 − $7,500 = **$900**  
(c) Current IBNR: AY 2021 = $9,400 − $8,900 = **$500**; AY 2022 = $10,200 − $8,400 = **$1,800**  
(d) Total needed = Actual emergence + Current IBNR:  
AY 2021: $700 + $500 = $1,200 needed vs. $900 prior → **deficiency of $300**  
AY 2022: $900 + $1,800 = $2,700 needed vs. $2,000 prior → **deficiency of $700**  
(e) Both AYs are deficient — prior estimates were **systematically inadequate**. This suggests the selected LDFs or a priori were too low. The actuary should investigate whether claims handling changes, legal environment shifts, or inadequate development factors caused the deficiency.

**Problem 2:**  
(a) Expected emergence = $7,200 × (1/2.400 − 1/1.600) × ... wait, use the correct formula:  
% emerging from 24→36 = 1/CDF(36) − 1/CDF(24) = 1/1.600 − 1/2.400 = 0.625 − 0.417 = 0.208  
Expected = $7,200 × 0.208 = **$1,500**  
(b) A/E = $1,950 / $1,500 = **1.30**  
(c) **Adverse** — actual emergence is 30% higher than expected. The actuary should investigate: (1) Are there large adverse claims driving this? (2) Have settlement patterns changed? (3) Is the selected CDF from 24 to 36 months understated?

**Problem 3:**  
(a) Three explanations for rising R/P:  
① **Lengthening tail** — claims are taking longer to develop; investigate by comparing recent LDFs to historical.  
② **Premium decline** — premium adequacy has worsened; investigate by comparing loss ratios and rate levels.  
③ **Reserve strengthening** — the company was previously underreserved; investigate prior year A/E ratios.  

(b) At 24-month development (CDF = 2.5), only 40% of losses are reported. The chain ladder gives heavy weight to the observed loss level × 2.5, which amplifies any random variation in early reports — this is highly **leveraged** and unreliable for an immature year. The BF method uses the a priori loss estimate and blends it with the actual, reducing this leverage. For AY 2023 at 24 months, give **more weight to the BF estimate** ($10.8M). The chain ladder may be overstating the adverse development by amplifying an early high-severity claim or reporting surge that doesn't represent the true ultimate.

</details>